# Step 4 - Build an Improved Reconstruction Model

The goal is to reconstruct missing ADS-B trajectory segments better than the naive **great-circle baseline** (direct straight-line interpolation between gap endpoints).

Two improved methods are built and compared:

| Method | Type | Key idea |
|--------|------|----------|
| **Kalman Filter** | Physics-based | Constant-velocity state estimator warmed up on pre-gap ADS-B |
| **GRU Model** | Learned | Two-encoder BiGRU (before + after context) that predicts a residual correction on top of the great-circle baseline |

Evaluation uses ADS-C waypoints as ground truth. Neither method sees ADS-C during reconstruction.

## 1 - Setup

In [3]:
import sys, os, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NB_DIR = Path(r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\notebook')
sys.path.insert(0, str(NB_DIR))

from step3_baseline import reconstruct_gap_baseline
from step5_kalman_aeroml3 import reconstruct_single_kalman
from step5_train_gru import TrajectoryGRU, gc_interpolate_batch

# -- Resolve noel_part directory ------------------------------------------
NOEL_DIR = Path(r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\noel_part')
if not NOEL_DIR.exists():
    for _c in [Path('../noel_part'), Path('../../noel_part')]:
        if _c.resolve().exists():
            NOEL_DIR = _c.resolve(); break
os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

CLEAN_DIR = NOEL_DIR / 'cleaned_data_final'
OUT_DIR   = NOEL_DIR.parent / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]
print(f'Working dir : {os.getcwd()}')
print(f'Flights available: {len(flights)}')

# -- Load sample flight ----------------------------------------------------
FLIGHT_NAME = '20230709_a67811_122827_132249'
FLIGHT_DIR  = next((s / FLIGHT_NAME for s in CLEAN_DIR.iterdir()
                    if s.is_dir() and (s / FLIGHT_NAME).exists()), None)
if FLIGHT_DIR is None:
    FLIGHT_DIR = flights[0]
    print(f'Fallback flight: {FLIGHT_DIR.name}')
else:
    print(f'Flight: {FLIGHT_NAME}')

bef  = pd.read_parquet(FLIGHT_DIR / 'adsb_before.parquet')
aft  = pd.read_parquet(FLIGHT_DIR / 'adsb_after.parquet')
adsc = pd.read_parquet(FLIGHT_DIR / 'adsc.parquet')

for df in [bef, aft, adsc]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_localize(None)
    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

t_gap_start = bef['timestamp'].iloc[-1]
t_gap_end   = aft['timestamp'].iloc[0]
bef_trim = bef[bef['timestamp'] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
aft_trim = aft[aft['timestamp'] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)

gap_min  = (t_gap_end - t_gap_start).total_seconds() / 60
adsc_gap = adsc[(adsc['timestamp'] >= t_gap_start) & (adsc['timestamp'] <= t_gap_end)].reset_index(drop=True)

print(f'Gap duration : {gap_min:.1f} min')
print(f'Before points: {len(bef_trim)}')
print(f'After points : {len(aft_trim)}')
print(f'ADS-C waypoints in gap: {len(adsc_gap)}')


Working dir : C:\Users\marko\Desktop\AeroML3(og)\AeroML3\noel_part
Flights available: 2149
Flight: 20230709_a67811_122827_132249
Gap duration : 212.0 min
Before points: 121
After points : 121
ADS-C waypoints in gap: 219


## 2 - Baseline: Great-Circle Interpolation

The **baseline** draws a straight great-circle arc from the last known ADS-B point before the gap to the first known point after it, with linearly interpolated altitude.

This is the naive method we aim to beat - it requires no physics model and no training data.

In [4]:
# Run baseline great-circle interpolation
recon_base = reconstruct_gap_baseline(bef_trim, aft_trim)
print(f'Baseline reconstruction: {len(recon_base)} steps')
print(f"Start: lat={recon_base['latitude'].iloc[0]:.3f}  lon={recon_base['longitude'].iloc[0]:.3f}")
print(f"End  : lat={recon_base['latitude'].iloc[-1]:.3f}  lon={recon_base['longitude'].iloc[-1]:.3f}")
display(recon_base[['timestamp','latitude','longitude','altitude']].head(5))


Baseline reconstruction: 848 steps
Start: lat=55.004  lon=-14.201
End  : lat=48.186  lon=-61.610


,timestamp,latitude,longitude,altitude
0,2023-07-09 12:02:45,55.003570,-14.200898,10972.800000
1,2023-07-09 12:03:00,55.007675,-14.261434,10973.519717
2,2023-07-09 12:03:15,55.011749,-14.321983,10974.239433
3,2023-07-09 12:03:30,55.015793,-14.382544,10974.959150
4,2023-07-09 12:03:45,55.019807,-14.443117,10975.678867


## 3 - Kalman Filter

The Kalman filter is a physics-based approach. It requires no training data.

### How it works
1. **Warmup** - runs through the last 15 ADS-B points before the gap estimating position and velocity
2. **Override velocity** - sets velocity to point directly toward the gap endpoint, preventing drift
3. **Forward predict** - propagates the state forward using a constant-velocity motion model
4. **Retime** - rescales timestamps so the aircraft moves at constant speed

### State vector
`x = [lat, lon, alt, dlat/s, dlon/s, dalt/s]`

### Key matrices
- **F** - state transition (position += velocity x dt)
- **Q** - process noise (uncertainty in acceleration)
- **R** - measurement noise (ADS-B position accuracy)

In [5]:
# Run Kalman smoother
recon_kalman = reconstruct_single_kalman(bef_trim, aft_trim)
print(f'Kalman reconstruction: {len(recon_kalman)} steps')
print(f"Start: lat={recon_kalman['latitude'].iloc[0]:.3f}  lon={recon_kalman['longitude'].iloc[0]:.3f}")
print(f"End  : lat={recon_kalman['latitude'].iloc[-1]:.3f}  lon={recon_kalman['longitude'].iloc[-1]:.3f}")
display(recon_kalman[['timestamp','latitude','longitude','altitude']].head(5))


Kalman reconstruction: 849 steps
Start: lat=54.999  lon=-14.140
End  : lat=48.168  lon=-61.655


,timestamp,latitude,longitude,altitude
0,2023-07-09 12:02:30,54.999435,-14.140373,10972.800000
1,2023-07-09 12:02:45,55.000893,-14.201547,10973.518868
2,2023-07-09 12:03:00,55.002327,-14.262731,10974.237736
3,2023-07-09 12:03:15,55.003742,-14.323920,10974.956604
4,2023-07-09 12:03:30,55.005135,-14.385107,10975.675472


## 4 - GRU Model

The GRU is a learned sequence model trained in `step5_train_gru.py`, with weights saved to `step5/best_model.pt`.

### Architecture
```
Before context  (64 steps x 6 features) -> Bidirectional GRU  (H=128, L=2)
After context   (32 steps x 6 features) -> Unidirectional GRU (H=128, L=2)
                        v
               Context vector (2H + H + gap_norm)
                        v
              Decoder MLP  ->  residual (deltalat, deltalon) at each tau
                        v
       Prediction = great-circle baseline(tau) + residual
```

### Input features (6 per timestep, resampled at 60 s)
`lat_norm`, `lon_norm`, `vel_norm`, `hdg_sin`, `hdg_cos`, `alt_norm`

### Key design choices
- **Two encoders** - one for the ADS-B segment before the gap, one for the segment after
- **Residual prediction** - corrects the great-circle baseline rather than predicting absolute position
- **Tau-based output** - predicts at any fractional position along the gap, not just fixed steps
- **Test-set result** (291 flights): GRU **90 km** mean error vs Baseline **140 km** (35% improvement)

## 4b - Run GRU on Sample Flight

In [6]:
import json as _json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_PATH = r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\step5\best_model.pt'
NORM_PATH  = r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\artifacts\step4_ml_dataset\dataset\normalization_stats.json'

with open(NORM_PATH) as f:
    stats = _json.load(f)

BEFORE_STEPS = stats['before_steps']
AFTER_STEPS  = stats['after_steps']
RESAMPLE_SEC = stats['resample_sec']

model_gru = TrajectoryGRU().to(device)
model_gru.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model_gru.eval()
print('TrajectoryGRU loaded')
print(f'  before_steps={BEFORE_STEPS}, after_steps={AFTER_STEPS}, resample={RESAMPLE_SEC}s')

def _normalize_seq(df):
    lat = (df['latitude'].values  - stats['lat']['mean']) / stats['lat']['std']
    lon = (df['longitude'].values - stats['lon']['mean']) / stats['lon']['std']
    vel = (df['velocity_mps'].values - stats['vel']['mean']) / stats['vel']['std']
    hdg = np.radians(df['heading_deg'].values)
    alt = (df['altitude'].values  - stats['alt']['mean']) / stats['alt']['std']
    return np.stack([lat, lon, vel, np.sin(hdg), np.cos(hdg), alt], axis=1).astype(np.float32)

def _resample_df(df, freq_sec):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = (df.dropna(subset=['latitude','longitude','altitude','velocity_mps','heading_deg'])
            .sort_values('timestamp').drop_duplicates('timestamp').set_index('timestamp'))
    cols = ['latitude','longitude','altitude','velocity_mps','heading_deg']
    grid = pd.date_range(df.index[0], df.index[-1], freq=f'{freq_sec}s')
    out  = df[cols].reindex(df[cols].index.union(grid)).interpolate('time').loc[grid]
    return out.reset_index().rename(columns={'index':'timestamp'})

def _prepare_seq(df, n_steps, from_end):
    arr = _normalize_seq(df)
    arr = arr[-n_steps:] if from_end else arr[:n_steps]
    n_valid = arr.shape[0]
    pad = np.zeros((n_steps - n_valid, arr.shape[1]), dtype=np.float32)
    if from_end:
        arr  = np.vstack([pad, arr])
        mask = np.array([0.0]*(n_steps - n_valid) + [1.0]*n_valid, dtype=np.float32)
    else:
        arr  = np.vstack([arr, pad])
        mask = np.array([1.0]*n_valid + [0.0]*(n_steps - n_valid), dtype=np.float32)
    return arr, mask

# -- Check required columns exist ------------------------------------------
required_cols = ['latitude','longitude','altitude','velocity_mps','heading_deg']
for col in required_cols:
    if col not in bef_trim.columns:
        raise ValueError(f'Missing column in bef_trim: {col}')

bef_rs = _resample_df(bef_trim, RESAMPLE_SEC)
aft_rs = _resample_df(aft_trim, RESAMPLE_SEC)

bef_arr, bef_mask = _prepare_seq(bef_rs, BEFORE_STEPS, from_end=True)
aft_arr, aft_mask = _prepare_seq(aft_rs, AFTER_STEPS,  from_end=False)

dt    = 15.0
t0    = bef_trim['timestamp'].iloc[-1]
t1    = aft_trim['timestamp'].iloc[0]
gap_s = (t1 - t0).total_seconds()
n_out = max(1, int(round(gap_s / dt)))
taus  = np.array([(i + 1) / (n_out + 1) for i in range(n_out)], dtype=np.float32)

la0 = np.array([float(bef_trim['latitude'].iloc[-1])],  dtype=np.float32)
lo0 = np.array([float(bef_trim['longitude'].iloc[-1])], dtype=np.float32)
la1 = np.array([float(aft_trim['latitude'].iloc[0])],   dtype=np.float32)
lo1 = np.array([float(aft_trim['longitude'].iloc[0])],  dtype=np.float32)
bl_lat, bl_lon = gc_interpolate_batch(la0, lo0, la1, lo1, taus[None, :])

def _t(arr): return torch.FloatTensor(arr[None]).to(device)

batch = {
    'before_seq':   _t(bef_arr),
    'before_mask':  _t(bef_mask),
    'after_seq':    _t(aft_arr),
    'after_mask':   _t(aft_mask),
    'gap_norm':     torch.FloatTensor([gap_s / 6000.0]).to(device),
    'adsc_tau':     torch.FloatTensor(taus[None]).to(device),
    'baseline_lat': torch.FloatTensor(bl_lat.astype(np.float32)).to(device),
    'baseline_lon': torch.FloatTensor(bl_lon.astype(np.float32)).to(device),
}

with torch.no_grad():
    pred_lat, pred_lon = model_gru(batch)

pred_lat = pred_lat.cpu().numpy()[0]
pred_lon = pred_lon.cpu().numpy()[0]

alt0 = float(bef_trim['altitude'].iloc[-1])
alt1 = float(aft_trim['altitude'].iloc[0])
pred_alt = np.linspace(alt0, alt1, n_out)

timestamps = [t0 + pd.Timedelta(seconds=dt * (i + 1)) for i in range(n_out)]

recon_gru = pd.DataFrame({
    'timestamp':    timestamps,
    'latitude':     pred_lat,
    'longitude':    pred_lon,
    'altitude':     pred_alt,
})

print(f'GRU reconstruction: {len(recon_gru)} steps')
print(f"Start: lat={recon_gru['latitude'].iloc[0]:.3f}  lon={recon_gru['longitude'].iloc[0]:.3f}")
print(f"End  : lat={recon_gru['latitude'].iloc[-1]:.3f}  lon={recon_gru['longitude'].iloc[-1]:.3f}")
display(recon_gru[['timestamp','latitude','longitude','altitude']].head())


TrajectoryGRU loaded
  before_steps=64, after_steps=32, resample=60s
GRU reconstruction: 848 steps
Start: lat=55.037  lon=-14.354
End  : lat=48.409  lon=-61.563


,timestamp,latitude,longitude,altitude
0,2023-07-09 12:02:45,55.037079,-14.353806,10972.800000
1,2023-07-09 12:03:00,55.041908,-14.413808,10973.519717
2,2023-07-09 12:03:15,55.046715,-14.473816,10974.239433
3,2023-07-09 12:03:30,55.051491,-14.533834,10974.959150
4,2023-07-09 12:03:45,55.056240,-14.593860,10975.678867


## 5 - Comparison: Baseline vs Kalman vs GRU

All three reconstructions are evaluated against ADS-C waypoints (ground truth not used during reconstruction).

**Metric**: mean closest-point distance (km) - for each ADS-C waypoint, find the nearest reconstructed point.

In [7]:
def error_km(recon_df, truth_df):
    if len(truth_df) == 0 or len(recon_df) == 0:
        return float('nan')
    lat1 = np.radians(truth_df['latitude'].values[:, None])
    lon1 = np.radians(truth_df['longitude'].values[:, None])
    lat2 = np.radians(recon_df['latitude'].values[None, :])
    lon2 = np.radians(recon_df['longitude'].values[None, :])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return float((2 * 6_371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))).min(axis=1).mean())

print(f'ADS-C ground-truth waypoints : {len(adsc_gap)}')
print(f'Gap duration                 : {gap_min:.1f} min')
print()

base_err   = error_km(recon_base,   adsc_gap)
kalman_err = error_km(recon_kalman, adsc_gap)
gru_err    = error_km(recon_gru,    adsc_gap)

print(f"{'Method':<35} {'Mean error':>12}  {'vs Baseline':>12}")
print('-' * 62)
print(f"  {'Baseline (great-circle)':<33} {base_err:>9.1f} km   {'-':>12}")
print(f"  {'Kalman filter':<33} {kalman_err:>9.1f} km   {(1-kalman_err/base_err)*100:>+10.1f}%")
print(f"  {'GRU (step5/best_model.pt)':<33} {gru_err:>9.1f} km   {(1-gru_err/base_err)*100:>+10.1f}%")

best = min([('Baseline', base_err), ('Kalman', kalman_err), ('GRU', gru_err)], key=lambda x: x[1])
print(f'Best on this flight: {best[0]} ({best[1]:.1f} km)')


ADS-C ground-truth waypoints : 219
Gap duration                 : 212.0 min

Method                                Mean error   vs Baseline
--------------------------------------------------------------
  Baseline (great-circle)                93.1 km              -
  Kalman filter                          51.5 km        +44.7%
  GRU (step5/best_model.pt)             119.5 km        -28.4%
Best on this flight: Kalman (51.5 km)


## 6 - Notes

- Full aggregate evaluation across all 2149 flights is in **`05_evaluate.ipynb`**
- To retrain the GRU, run `step5_train_gru.py` (dataset prepared by `04_step4_dataset.ipynb`)
- The GRU predicts at arbitrary tau positions, so the single-flight comparison above may vary - the 35% improvement is the aggregate result on 291 held-out test flights